# 99 · 综合项目（Capstone）：轮到你了

> **Part 9 · 收尾。开放式练习——没有标准答案。**

你已经造好了 minitorch 全套零件。现在挑一个项目**亲手做做看**，把所学融会贯通。下面给出脚手架与提示，但**实现留给你**。

## 推荐项目（任选其一）

1. **mini-GPT（推荐）**：用 `causal_mask` 把编码器变成 **decoder-only** 的字符级语言模型，训练它续写文本。（脚手架见下。）
2. **新优化器**：实现 `AdamW`（解耦权重衰减）或 `AdaGrad`，在 nb11 的损失面上和现有优化器对比轨迹。
3. **新层**：实现 `GRU`（比 LSTM 更简洁的门控单元）或 `GELU` 激活或 `AvgPool2d`，并写 gradcheck 测试。
4. **真实任务**：用你的 CNN 在 MNIST 全量数据上训练到 99%，或换一个数据集试试。

## 脚手架：mini-GPT（字符级语言模型）

目标：给定一段文本，训练模型预测"下一个字符"，然后从一个起始字符**自回归生成**新文本。这正是 GPT 的最小内核。

下面的数据准备是**可运行的**；模型部分给出骨架与 `TODO`，由你补全。

In [ ]:
import numpy as np
import minitorch
from minitorch import Tensor, nn, no_grad
from minitorch.functional import cross_entropy
from minitorch.nn import causal_mask

# ---- 可运行：准备字符级数据 ----
text = ("to be or not to be that is the question "
        "whether tis nobler in the mind to suffer ") * 4
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
V = len(chars)
data = np.array([stoi[c] for c in text])
print(f"文本长度 {len(text)}，词表大小 V={V}")

def get_batch(seq, block=16, batch=32, seed=None):
    rng = np.random.RandomState(seed)
    ix = rng.randint(0, len(seq) - block - 1, size=batch)
    X = np.stack([seq[i:i+block] for i in ix])
    Y = np.stack([seq[i+1:i+block+1] for i in ix])   # 目标 = 右移一位
    return X, Y

Xb, Yb = get_batch(data)
print("一个 batch:", Xb.shape, "->", Yb.shape)

### 你的任务：补全 mini-GPT

把下面的骨架复制到一个新代码单元，完成所有 `TODO`：

```python
class MiniGPT(nn.Module):
    def __init__(self, V, block, d=64, heads=4, layers=2):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.pos = nn.PositionalEncoding(d)
        self.blocks = [nn.TransformerEncoderLayer(d, heads, 4*d) for _ in range(layers)]
        self.head = nn.Linear(d, V)
        self.block = block

    def forward(self, idx):           # idx: (B, T) 整数
        x = self.pos(self.emb(idx))
        mask = causal_mask(idx.shape[1])     # TODO: 关键！用因果掩码，不能看未来
        for blk in self.blocks:
            x = blk(x, mask)          # TODO: 把 mask 传进每个编码层
        return self.head(x)           # (B, T, V)

# 训练循环（TODO：补全 forward/loss/backward/step）
model = MiniGPT(V, block=16)
opt = minitorch.optim.Adam(model.parameters(), lr=3e-3)
for step in range(300):
    Xb, Yb = get_batch(data, block=16, batch=32)
    opt.zero_grad()
    logits = model(Xb)                       # (B, T, V)
    B, T, _ = logits.shape
    loss = cross_entropy(logits.reshape(B*T, V), Yb.reshape(B*T))   # TODO
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(step, float(loss.data))
```

### 自回归生成（也请你补全）

```python
def generate(model, start, n=80):
    idx = [stoi[c] for c in start]
    for _ in range(n):
        context = np.array(idx[-16:])[None, :]   # 取最近 block 个字符
        with no_grad():
            logits = model(context).data[0, -1]  # 最后一个位置的预测
        idx.append(int(logits.argmax()))         # TODO: 也可改成按概率采样，更有创造性
    return "".join(itos[i] for i in idx)

print(generate(model, "to be"))
```

## 调试锦囊（非常有用）

1. **先过拟合一个小 batch**：拿**一个**固定 batch 反复训练，如果损失不能降到接近 0，说明实现有 bug（而非调参问题）。
2. **gradcheck 新算子**：每实现一个新层/激活，先用 `minitorch.gradcheck` 或数值梯度核对再用。
3. **看形状**：报错十有八九是形状不匹配，多 `print(x.shape)`。
4. **对照 PyTorch**：拿同样输入喂 PyTorch 等价模块，比对前向输出与梯度。
5. **从小做起**：先把模型/数据都调到最小能跑通，再逐步放大。

## 提交你的成果

完成后，不妨把你的实现整理成一个新 notebook 放进 `notebooks/`，或给 `minitorch` 加一个新模块 + 对应的 `tests/`。**真正的理解，来自亲手把它做出来。**

祝你玩得开心，学得扎实！🎓